In [1]:
import os
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/repo
!pip uninstall ultralytics -y -q
!git clone https://github.com/AadeeshRS/road-damage-detection-thesis.git /kaggle/working/repo
!pip install -e /kaggle/working/repo/ultralytics -q
!pip install sahi -q
print("Setup done!")

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (1115/1115), done.
remote: Compressing objects: 100% (862/862), done.
remote: Total 1511 (delta 281), reused 1062 (delta 237), pack-reused 396 (from 3)
Receiving objects: 100% (1511/1511), 478.42 MiB | 42.73 MiB/s, done.
Resolving deltas: 100% (342/342), done.
Updating files: 100% (1192/1192), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.9 MB/s eta 0:00:00
Setup done!


In [2]:
import sys
for key in list(sys.modules.keys()):
    if 'ultralytics' in key:
        del sys.modules[key]
sys.path.insert(0, '/kaggle/working/repo/ultralytics')

import torch
from ultralytics import YOLO
print("CUDA:", torch.cuda.is_available())

model = YOLO('/kaggle/working/repo/ultralytics/ultralytics/cfg/models/v8/yolov8m-full-hybrid.yaml')
print("SUCCESS! Params:", f"{sum(p.numel() for p in model.model.parameters()):,}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA: True
LEGACY = True
SUCCESS! Params: 23,320,583


In [3]:
yaml_content = """
path: /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT

train: train/images
val: val/images
test: test/images

names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: other_corruption
  4: pothole
"""
with open("/kaggle/working/dataset.yaml", "w") as f:
    f.write(yaml_content)
print("dataset.yaml created")


dataset.yaml created


In [4]:
import os
os.chdir('/kaggle/working/repo')
!git pull origin main


remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 7 (delta 3), reused 7 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 183.69 KiB | 14.13 MiB/s, done.
From https://github.com/AadeeshRS/road-damage-detection-thesis
 * branch            main       -> FETCH_HEAD
   7386471..28183f3  main       -> origin/main
Updating 7386471..28183f3
Fast-forward
 ultralytics/ultralytics/assets/bus.jpg    | Bin 0 -> 137419 bytes
 ultralytics/ultralytics/assets/zidane.jpg | Bin 0 -> 50427 bytes
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ultralytics/ultralytics/assets/bus.jpg
 create mode 100644 ultralytics/ultralytics/assets/zidane.jpg


In [5]:
import os
import glob

dataset_path = "/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT"
train_india = [f for f in glob.glob(f"{dataset_path}/train/images/*.jpg") if os.path.basename(f).startswith("India_")]
val_india = [f for f in glob.glob(f"{dataset_path}/val/images/*.jpg") if os.path.basename(f).startswith("India_")]
with open("/kaggle/working/train_india.txt", "w") as f:
    f.write("\n".join(train_india))
with open("/kaggle/working/val_india.txt", "w") as f:
    f.write("\n".join(val_india))


In [6]:
yaml_india = """
path: /kaggle/working
train: train_india.txt
val: val_india.txt
names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: other_corruption
  4: pothole
"""
with open("/kaggle/working/dataset_india.yaml", "w") as f:
    f.write(yaml_india)
print(f"India subset created: {len(train_india)} train, {len(val_india)} val")

India subset created: 5368 train, 1172 val


In [7]:
model = YOLO('/kaggle/working/repo/ultralytics/ultralytics/cfg/models/v8/yolov8m-full-hybrid.yaml')
results = model.train(
    data="/kaggle/working/dataset_india.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    workers=4,
    project="thesis_experiments",
    name="full_hybrid_india_subset"
)

LEGACY = True
New https://pypi.org/project/ultralytics/8.4.103 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_india.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/repo/ultr

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      7.55G      3.511      5.002       3.59         13        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:330.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 1.6it/s 23.3s0.5ss
                   all       1172       1249   0.000575      0.212    0.00177   0.000436

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      7.49G      3.157       4.25      2.935         14        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:270.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.6s0.5s
                   all       1172       1249    0.00105       0.23    0.00412      0.001

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      28/30      7.49G      1.906      2.559      1.969          7        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:280.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.6s0.5s
                   all       1172       1249      0.458      0.255      0.203     0.0904

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      7.49G      1.895      2.541      1.963          5        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:280.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.7s0.5s
                   all       1172       1249      0.458      0.261      0.209     0.0913

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      7.49G      1.891      2.503      1.951         12        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:280.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.6s0.5s
                   all       1172       1249      0.463      0.264      0.213     0.0955

30 epochs completed in 1.896 hours.
Optimizer stripped from /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/weights/last.pt, 47.0MB
Optimizer stripped from /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/weights/best.pt, 47.0MB

Validating /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/weights/best.pt...
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8m-full-hybrid summary: 149 layers, 23,305,559 parameters, 0 gradients, 67.0 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)

In [8]:
!zip -r /kaggle/working/full_hybrid_india_subset_results.zip /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset


  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/ (stored 0%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/train_batch0.jpg (deflated 3%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/val_batch2_labels.jpg (deflated 10%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/BoxF1_curve.png (deflated 12%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/train_batch6721.jpg (deflated 13%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/val_batch1_labels.jpg (deflated 10%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/confusion_matrix_normalized.png (deflated 21%)
  adding: kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_subset/results.csv (deflated 60%)
  adding: kaggle/working/repo/runs/detect/thesis_exper

In [2]:
import os
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/repo
!pip uninstall ultralytics -y -q
!git clone https://github.com/AadeeshRS/road-damage-detection-thesis.git /kaggle/working/repo
!pip install -e /kaggle/working/repo/ultralytics -q
print("Custom Hybrid repo installed!")


Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (1115/1115), done.
remote: Compressing objects: 100% (862/862), done.
remote: Total 1511 (delta 281), reused 1062 (delta 237), pack-reused 396 (from 3)
Receiving objects: 100% (1511/1511), 478.42 MiB | 43.06 MiB/s, done.
Resolving deltas: 100% (342/342), done.
Updating files: 100% (1192/1192), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
Custom Hybrid repo installed!


In [3]:
import sys
for key in list(sys.modules.keys()):
    if 'ultralytics' in key:
        del sys.modules[key]
sys.path.insert(0, '/kaggle/working/repo/ultralytics')

import torch
from ultralytics import YOLO

print("CUDA Available:", torch.cuda.is_available())
import ultralytics
print("Loaded from:", ultralytics.__file__)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA Available: True
Loaded from: /kaggle/working/repo/ultralytics/ultralytics/__init__.py


In [4]:
import os
import glob

print("Locating RDD_SPLIT dataset...")

# 1. Fast search for the RDD_SPLIT folder
dataset_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "RDD_SPLIT" in dirs:
        dataset_path = os.path.join(root, "RDD_SPLIT")
        break

assert dataset_path is not None, "Error: Could not find RDD_SPLIT. Ensure the RDD 2022 dataset is added."
print(f"Found Dataset at: {dataset_path}")

# 2. Fast exact filtering
train_india = [f for f in glob.glob(f"{dataset_path}/train/images/*.jpg") if os.path.basename(f).startswith("India_")]
val_india = [f for f in glob.glob(f"{dataset_path}/val/images/*.jpg") if os.path.basename(f).startswith("India_")]

print(f"India Train Images: {len(train_india)}")
print(f"India Val Images: {len(val_india)}")

assert len(train_india) > 0, "Error: No India training images found."

# 3. Write image list files
with open("/kaggle/working/train_india.txt", "w") as f:
    f.write("\n".join(train_india))
with open("/kaggle/working/val_india.txt", "w") as f:
    f.write("\n".join(val_india))

# 4. Write dataset.yaml
yaml_india = """
path: /kaggle/working
train: train_india.txt
val: val_india.txt

names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: other_corruption
  4: pothole
"""
with open("/kaggle/working/dataset_india.yaml", "w") as f:
    f.write(yaml_india)

print("SUCCESS! dataset_india.yaml created.")


Locating RDD_SPLIT dataset...
Found Dataset at: /kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT
India Train Images: 5368
India Val Images: 1172
SUCCESS! dataset_india.yaml created.


In [5]:
model = YOLO('/kaggle/working/repo/ultralytics/ultralytics/cfg/models/v8/yolov8m-full-hybrid.yaml')
print(f"Full Hybrid Model Loaded! Total Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

results = model.train(
    data="/kaggle/working/dataset_india.yaml",
    epochs=100,                                 
    imgsz=640,
    batch=16,
    workers=4,
    project="thesis_experiments",
    name="full_hybrid_india_100e"               
)


LEGACY = True
Full Hybrid Model Loaded! Total Parameters: 23,320,583
New https://pypi.org/project/ultralytics/8.4.104 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_india.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      7.55G      3.508      4.993      3.587         13        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:320.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 1.7it/s 22.1s0.5ss
                   all       1172       1249   0.000908      0.231    0.00134   0.000342

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      7.49G      3.099      4.261      2.948         14        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:260.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.3s0.5s
                   all       1172       1249   0.000889       0.21    0.00379    0.00105

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

     98/100      7.49G      1.609      1.808      1.716          4        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:260.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.4s0.5s
                   all       1172       1249      0.396      0.322      0.324      0.144

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      7.49G      1.596      1.796       1.71         13        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:260.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.5s0.5s
                   all       1172       1249      0.408       0.32      0.326      0.144

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    100/100      7.49G      1.594      1.781      1.708          8        640: 100% ━━━━━━━━━━━━ 336/336 1.6it/s 3:250.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 2.0it/s 18.4s0.5s
                   all       1172       1249      0.405      0.326      0.325      0.144

100 epochs completed in 6.257 hours.
Optimizer stripped from /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_100e/weights/last.pt, 47.0MB
Optimizer stripped from /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_100e/weights/best.pt, 47.0MB

Validating /kaggle/working/repo/runs/detect/thesis_experiments/full_hybrid_india_100e/weights/best.pt...
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8m-full-hybrid summary: 149 layers, 23,305,559 parameters, 0 gradients, 67.0 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100

In [6]:
!cd /kaggle/working/repo/runs/detect/thesis_experiments && zip -r /kaggle/working/full_hybrid_india_100e_results.zip full_hybrid_india_100e

  adding: full_hybrid_india_100e/ (stored 0%)
  adding: full_hybrid_india_100e/val_batch1_labels.jpg (deflated 10%)
  adding: full_hybrid_india_100e/train_batch30241.jpg (deflated 8%)
  adding: full_hybrid_india_100e/train_batch2.jpg (deflated 2%)
  adding: full_hybrid_india_100e/val_batch0_labels.jpg (deflated 10%)
  adding: full_hybrid_india_100e/val_batch0_pred.jpg (deflated 10%)
  adding: full_hybrid_india_100e/results.png (deflated 8%)
  adding: full_hybrid_india_100e/confusion_matrix.png (deflated 22%)
  adding: full_hybrid_india_100e/train_batch30242.jpg (deflated 12%)
  adding: full_hybrid_india_100e/weights/ (stored 0%)
  adding: full_hybrid_india_100e/weights/best.pt (deflated 8%)
  adding: full_hybrid_india_100e/weights/last.pt (deflated 8%)
  adding: full_hybrid_india_100e/val_batch2_pred.jpg (deflated 10%)
  adding: full_hybrid_india_100e/train_batch1.jpg (deflated 4%)
  adding: full_hybrid_india_100e/train_batch0.jpg (deflated 3%)
  adding: full_hybrid_india_100e/args.yam